In [ ]:
import requests
import matplotlib.pyplot as plt
import pandas as pd
import json
import csv
import numpy as np

In [ ]:
%pip install google-analytics-data pandas

# Google Analytics API

In [ ]:
# PROPERTY ID & KEY FILE

from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.oauth2 import service_account

PROPERTY_ID = "321460044"

KEY_FILE = r"C:\Users\jlmow\Documents-C Drive\NSS-C Drive\Capstone\sfs-mrktg-76749b6efce7.json"

credentials = service_account.Credentials.from_service_account_file(
    KEY_FILE
)

client = BetaAnalyticsDataClient(credentials=credentials)

print("Connection setup completed.")

## API Pull: GA1 - Dates, Sessions, Source/Medium

In [ ]:
from datetime import date, timedelta

# First day of the current month
first_day_this_month = date.today().replace(day=1)

# Last day of the previous month
last_day_last_month = first_day_this_month - timedelta(days=1)

In [ ]:
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest
)
request = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[
        Dimension(name="date"),
        Dimension(name="sessionSourceMedium"),
        Dimension(name="sessionDefaultChannelGroup")
        ],
    metrics=[
        Metric(name="sessions")
    ],
    date_ranges=[
        DateRange(
            start_date="2022-08-01",
            end_date=last_day_last_month.strftime("%Y-%m-%d")
        )
    ],
 limit=100000
)
response_ga1 = client.run_report(request)

print("Number of rows returned:", len(response_ga1.rows))

response_ga1

In [ ]:
data = []

for row in response_ga1.rows:
    data.append({
        "date": row.dimension_values[0].value,
        "source_medium": row.dimension_values[1].value,
        "sessionDefaultChannelGroup": row.dimension_values[2].value,
        "sessions": row.metric_values[0].value
    })

ga1 = pd.DataFrame(data)

ga1["date"] = pd.to_datetime(
    ga1["date"],
    format="%Y%m%d"
)

ga1["sessions"] = pd.to_numeric(
    ga1["sessions"]

)

ga1.sort_values(by="date")

In [ ]:
ga1.dtypes

### Save DF to .csv

In [ ]:
ga1.to_csv('ga1.csv', index=False)

## API Pull: GA2 - Date, Sessions, Source/Medium, Page Paths

In [ ]:
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest
)
request = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[
        Dimension(name="date"),
        Dimension(name="sessionSourceMedium"),
        Dimension (name="sessionDefaultChannelGroup"),
        Dimension(name="pagePath")
    ],
    metrics=[
        Metric(name="sessions"),
        Metric(name="screenpageviews")
    ],
    date_ranges=[
        DateRange(
            start_date="2022-08-01",
            end_date=last_day_last_month.strftime("%Y-%m-%d")
        )
    ],
 limit=100000
)
response_ga2 = client.run_report(request)

print("Number of rows returned:", len(response_ga2.rows))

response_ga2

In [ ]:
data = []

for row in response_ga2.rows:
    data.append({
        "date": row.dimension_values[0].value,
        "source_medium": row.dimension_values[1].value,
        "sessionDefaultChannelGroup": row.dimension_values[2].value,
        "page_path": row.dimension_values[3].value,
        "sessions": row.metric_values[0].value,
        "pageviews": row.metric_values[1].value

    })

ga2 = pd.DataFrame(data)

ga2["date"] = pd.to_datetime(
    ga2["date"],
    format="%Y%m%d"
)

ga2["sessions"] = pd.to_numeric(
    ga2["sessions"]

)

ga2.sort_values(by="date")

In [ ]:
ga2['sessions'].sum()

In [ ]:
ga2.dtypes

### GA2 - Sum by source_medium

In [ ]:
ga2_date = ga2.groupby(['source_medium','date']).sum()
ga2_date

### GA2 - Sum by Date

In [ ]:
ga2_all = ga2.groupby(['date']).sum()
ga2
#sum with text concatenates

In [ ]:
ga2.to_csv('ga2.csv', index=False)

# HubSpot API

In [ ]:
%pip install requests pandas

In [ ]:
from getpass import getpass

HUBSPOT_TOKEN = getpass("Paste your HubSpot access token: ")

In [ ]:
import requests
import pandas as pd

headers = {
    "Authorization": f"Bearer {HUBSPOT_TOKEN}",
    "Content-Type": "application/json"
}

url = f"https://api.hubapi.com/crm/v3/lists/{LIST_ID}/memberships"

params = {
    "limit": 100
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

response.raise_for_status()

data = response.json()

data

### API Pull: HS Contacts / Lead Segment

In [ ]:
url = f"https://api.hubapi.com/crm/v3/lists/{LIST_ID}/memberships"

contact_ids = []
after = None

while True:

    params = {
        "limit": 100
    }

    if after is not None:
        params["after"] = after

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    response.raise_for_status()

    hs1 = response.json()

    for record in hs1.get("results", []):
        contact_ids.append(record["recordId"])

    paging = hs1.get("paging")

    if not paging or "next" not in paging:
        break

    after = paging["next"]["after"]

print(f"Contacts found: {len(contact_ids)}")

In [ ]:
properties = [
    "firstname",
    "lastname",
    "email",
    "phone",
    "jobtitle",
    "company",
    "lifecyclestage",
    "lead_source_1__c",
    "lead_created_date__c",
    "city",
    "state"
]

batch_url = "https://api.hubapi.com/crm/v3/objects/contacts/batch/read"

contacts = []

for i in range(0, len(contact_ids), 100):

    batch_ids = contact_ids[i:i + 100]

    body = {
        "properties": properties,
        "inputs": [
            {"id": contact_id}
            for contact_id in batch_ids
        ]
    }

    response = requests.post(
        batch_url,
        headers=headers,
        json=body
    )

    response.raise_for_status()

    hs1 = response.json()

    contacts.extend(hs1["results"])

print(f"Contacts downloaded: {len(contacts)}")

In [ ]:
rows = []

for contact in contacts:

    p = contact["properties"]

    rows.append({
        "contact_id": contact["id"],
        "first_name": p.get("firstname"),
        "last_name": p.get("lastname"),
        "email": p.get("email"),
        "phone": p.get("phone"),
        "job_title": p.get("jobtitle"),
        "company": p.get("company"),
        "lifecycle_stage": p.get("lifecyclestage"),
        "lead_source": p.get("lead_source_1__c"),
        "lead_created_date": p.get("lead_created_date__c"),
        "city": p.get("city"),
        "state": p.get("state")
    })

hs1_df = pd.DataFrame(rows)

hs1_df

### API Pull: HS Companies

In [ ]:
association_url = "https://api.hubapi.com/crm/v4/associations/contacts/companies/labels"

response = requests.get(
    association_url,
    headers=headers
)

response.raise_for_status()

association_labels = response.json()

association_labels

In [ ]:
PRIMARY_COMPANY_TYPE_ID = 1

In [ ]:
association_url = (
    "https://api.hubapi.com/crm/v4/"
    "associations/contacts/companies/batch/read"
)

contact_company_associations = []

for i in range(0, len(contact_ids), 1000):

    batch_ids = contact_ids[i:i + 1000]

    body = {
        "inputs": [
            {"id": contact_id}
            for contact_id in batch_ids
        ]
    }

    response = requests.post(
        association_url,
        headers=headers,
        json=body
    )

    response.raise_for_status()

    hs2 = response.json()

    contact_company_associations.extend(hs2["results"])

In [ ]:
association_labels

In [ ]:
primary_company_map = {}

for record in contact_company_associations:

    contact_id = str(record["from"]["id"])

    for company in record.get("to", []):

        for association_type in company.get("associationTypes", []):

            if association_type["typeId"] == PRIMARY_COMPANY_TYPE_ID:

                primary_company_map[contact_id] = str(company["toObjectId"])

In [ ]:
primary_company_map

In [ ]:
hs1_df["primary_company_id"] = (
    hs1_df["contact_id"]
    .astype(str)
    .map(primary_company_map)
    .astype("string")
)

In [ ]:
hs1_df[
    ["contact_id", "first_name", "last_name", "primary_company_id"]
].head()

In [ ]:
company_ids = (
    hs1_df["primary_company_id"]
    .dropna()
    .unique()
    .tolist()
)

company_name_map = {}

batch_url = "https://api.hubapi.com/crm/v3/objects/companies/batch/read"

for i in range(0, len(company_ids), 100):

    batch_ids = company_ids[i:i + 100]

    body = {
        "properties": ["name"],
        "inputs": [
            {"id": company_id}
            for company_id in batch_ids
        ]
    }

    response = requests.post(
        batch_url,
        headers=headers,
        json=body
    )

    response.raise_for_status()

    hs3 = response.json()

    for company in hs3["results"]:
        company_name_map[company["id"]] = company["properties"].get("name")

In [ ]:
hs1_df["primary_associated_company"] = (
    hs1_df["primary_company_id"]
    .map(company_name_map)
)

In [ ]:
hs1_df.dtypes

In [ ]:
hs1_df['lead_created_date'] = pd.to_datetime(hs1_df['lead_created_date'], errors='coerce')
hs1_df

In [ ]:
hs1_df.to_csv('hs1.csv', index=False)

In [ ]:
hs1_df_filter = hs1_df.dropna(subset=['lead_created_date'])

In [ ]:
hs1_df_filter

In [ ]:
hs1_df_filter_counts = hs1_df_filter[['lead_created_date','lead_source','contact_id']].groupby(['lead_created_date','lead_source']).count().reset_index()
hs1_df_filter_counts

In [ ]:
hs1_df_filter_counts.to_csv('hs1_df_filter_counts.csv', index=False)

### API Pull: HS Deals

In [ ]:
from getpass import getpass

HUBSPOT_TOKEN = getpass("Paste your HubSpot access token: ")

In [ ]:
import requests

url = "https://api.hubapi.com/crm/v3/objects/deals"

headers = {
    "Authorization": f"Bearer {HUBSPOT_TOKEN}"
}

all_deals = []
after = None

while True:

    params = {
        "limit": 100,
        "properties": ",".join([
            "dealname",
            "amount",
            "dealstage",
            "pipeline",
            "closedate",
            "createdate",
            "hubspot_owner_id",
            "became_opportunity_date__c",
            "x2_demo_scheduled_date__c",
            "x3_demo_completed_date__c",
            "x4_proposal_sent_date__c",
            "x5_closed_lost_date__c",
            "x5_closed_won_date__c",
            "amount",
            "amount_in_home_currency",
            "lead_source_1__c",
            "dealtype"
        ]),
        "associations": "contacts"
    }

    if after is not None:
        params["after"] = after

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    response.raise_for_status()

    data = response.json()

    all_deals.extend(data["results"])

    if "paging" in data and "next" in data["paging"]:
        after = data["paging"]["next"]["after"]
    else:
        break

In [ ]:
deals = []

for deal in all_deals:

    properties = deal["properties"]

    deals.append({
        "deal_id": deal["id"],
        "deal_name": properties.get("dealname"),
        "amount": properties.get("amount"),
        "deal_stage": properties.get("dealstage"),
        "pipeline": properties.get("pipeline"),
        "close_date": properties.get("closedate"),
        "create_date": properties.get("createdate"),
        "owner_id": properties.get("hubspot_owner_id"),
        "became_deal": properties.get("became_opportunity_date__c"),
        "demo_scheduled": properties.get("x2_demo_scheduled_date__c"),
        "demo_completed": properties.get("x3_demo_completed_date__c"),
        "proposal_sent": properties.get("x4_proposal_sent_date__c"),
        "closed_lost": properties.get("x5_closed_lost_date__c"),
        "closed_won": properties.get("x5_closed_won_date__c"),
        "amount": properties.get("amount"),
        "lead_source": properties.get("lead_source_1__c"),
        "dealtype": properties.get("dealtype")
    })

deals_df = pd.DataFrame(deals)

deals_df

In [ ]:
deals_df["create_date"]=pd.to_datetime(deals_df["create_date"])
#deals_df["close_date"]=pd.to_datetime(deals_df["close_date"],format="%Y-%m-%dT%H:%M:%SZ")
deals_df["demo_scheduled"]=pd.to_datetime(deals_df["demo_scheduled"])
deals_df["demo_completed"]=pd.to_datetime(deals_df["demo_completed"])
deals_df["proposal_sent"]=pd.to_datetime(deals_df["proposal_sent"])
deals_df["closed_won"]=pd.to_datetime(deals_df["closed_won"])

In [ ]:
deals_df.dtypes

In [ ]:
date_columns = [
    "create_date",
    "close_date",
    "demo_scheduled",
    "demo_completed",
    "proposal_sent",
    "closed_lost",
    "closed_won"]
for col in date_columns:
    deals_df['demo_scheduled'] = pd.to_datetime(deals_df['demo_scheduled'], errors="coerce")
    deals_df['demo_completed'] = pd.to_datetime(deals_df['demo_completed'], errors="coerce")
    deals_df['proposal_sent'] = pd.to_datetime(deals_df['proposal_sent'], errors="coerce")
    deals_df['closed_won'] = pd.to_datetime(deals_df['closed_won'], errors="coerce")
deals_df

In [ ]:
deals_df.to_csv('deals_df.csv', index=False)

In [ ]:
url = "https://api.hubapi.com/crm/v3/properties/deals"

response = requests.get(
    url,
    headers=headers
)

response.raise_for_status()

deal_properties = response.json()["results"]

deal_properties_df = pd.DataFrame(deal_properties)

deal_properties_df[["name", "label", "type"]]

In [ ]:
deal_properties_df.to_csv('deal_properties_df.csv', index=False)

## Deal Contact Associations

In [ ]:
deal_contacts = []

for deal in all_deals:

    deal_id = deal["id"]

    contacts = (
        deal
        .get("associations", {})
        .get("contacts", {})
        .get("results", [])
    )

    for contact in contacts:

        deal_contacts.append({
            "deal_id": deal_id,
            "contact_id": contact["id"]
        })

deal_contacts_df = pd.DataFrame(deal_contacts)

deal_contacts_df

In [ ]:
deals_and_contacts = deal_contacts_df.merge(deals_df, on='deal_id', how='outer').merge(hs1_df, on='contact_id', how='outer')
deals_and_contacts

In [ ]:
deals_and_contacts.to_csv('deals_and_contacts.csv', index=False)

In [ ]:
deals_and_contacts.loc[deals_and_contacts['closed_won'].notna()][['closed_won','lead_source_y']]

# Sales Funnel Build

## Counting Sessions Stage

In [ ]:
sessions = ga1[['source_medium', 'sessionDefaultChannelGroup','date','sessions']].groupby(['date','source_medium', 'sessionDefaultChannelGroup']).sum().reset_index()
sessions['stage'] = 'sessions'
sessions = sessions.rename(columns = {'source_medium':'lead_source', 'sessions':'count'})
sessions

#date	source_medium	sessionDefaultChannelGroup	sessions


## Counting Contacts Stage

In [ ]:
leads = hs1_df_filter_counts[['lead_source','lead_created_date','contact_id']].groupby(['lead_created_date','lead_source']).sum().reset_index()
leads['stage'] = 'lead'
leads = leads.rename(columns = {'lead_created_date':'date','contact_id':'count'})
leads

#lead_created_date	lead_source	contact_id


## Counting Deal Stages

In [ ]:
engaged_leads = deals_df[['lead_source','became_deal','deal_id']].groupby(['became_deal','lead_source']).count().reset_index()
engaged_leads['stage'] = 'engaged'
engaged_leads = engaged_leads.rename(columns = {'became_deal':'date','deal_id':'count'})
engaged_leads

In [ ]:
demos = deals_df[['lead_source','demo_completed','deal_id']].groupby(['demo_completed','lead_source']).count().reset_index()
demos['stage'] = 'demo'
demos = demos.rename(columns = {'demo_completed':'date','deal_id':'count'})
demos

In [ ]:
closed_won = deals_df[['lead_source','closed_won','deal_id']].groupby(['closed_won','lead_source']).count().reset_index()
closed_won['stage'] = 'closed won'
closed_won = closed_won.rename(columns = {'closed_won':'date','deal_id':'count'})
closed_won

In [ ]:
funnel = pd.concat(
    [sessions, leads, engaged_leads, demos, closed_won],
    ignore_index=True
)
funnel

In [ ]:
funnel.to_csv('funnel.csv',index=False)

In [ ]:
funnel['lead_source'].to_csv('funnel_sources.csv',index=False)

In [ ]:
mrktg_sources = pd.read_csv("funnel_sources_csv.csv")
mrktg_sources

In [ ]:
funnel_campaigns = pd.merge(
    funnel,
    mrktg_sources,
    on="lead_source",
    how="inner"
)
funnel_campaigns

In [ ]:
funnel_campaigns.to_csv('funnel_campaigns.csv',index=False)